In [ ]:
import tensorflow as tf
import os
import torch.nn as nn
import numpy as np
import torch

Implement (using existing code) an LSTM model
Train it (use holdout val) over PTB data set
Vary num units and num of stacked LSTM layers
Obtain a test perplexity of 80 or below (lower = better)

In [7]:
url = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/"
files = ["ptb.train.txt", "ptb.valid.txt", "ptb.test.txt"]

for f in files: 
    tf.keras.utils.get_file(f, url + f)

def load_data(filename):
    path = os.path.join(os.path.expanduser('~'), '.keras/datasets', filename)
    with open(path, 'r', encoding = 'utf-8') as f:
        return f.read().replace('\n', '<eos>')

train_text = load_data('ptb.train.txt')
valid_text = load_data('ptb.valid.txt')
test_text = load_data('ptb.test.txt')


In [10]:
import torch

In [27]:
#More LSTM can be seen in lecture 3
#Use AWD-LSTMSs
#Get unique tokens
tokens = train_text.split()
vocab = sorted(list(set(tokens)))

valid_tok = valid_text.split()
valid_vocab = sorted(list(set(valid_tok)))

#Create mappings
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for i, word in enumerate(vocab)}

vocab_size = 10000
seq_length = 38
step = 1

def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(0, len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(tokens, seq_length)
X_valid, y_valid = create_sequences(valid_tok, seq_length)

In [12]:
if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size) #converts word IDs to vectors. vocab_size = num of distinct tokens. Embed_size = how big each word vector is 
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)
    def forward(self, x, h): #x:(batch_size, seq_length), h: (h0, c0), h0: zeros in shape (num_layers, batch_size, hidden_size), c0: same
        x = self.embed(x)
        out, (h,c) = self.lstm(x, h)
        out = out.reshape(out.size(0)*out.size(1), out.size(2))
        out = self.linear(out)
        return out, (h, c)

embed_size = 128
hidden_size = 5
num_layers = 2
learning_rate = 0.001


model = LSTMModel(vocab_size, embed_size, hidden_size, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [ ]:


model.eval()

#predict
h0_predict = torch.zeros(num_layers, 1, hidden_size, , device=device)
c0_predict = torch.zeros(num_layers, 1, hidden_size, device = device)
val_per_time_test, _ =model(X_valid[0], (h0_predict, c0_predict))
print(val_per_time_test[-1])
id_for_last = torch.argmax(val_per_time_test[-1])
word_for_last = idx2word[id_for_last]
print(word_for_last)

aer


KeyError: 'I'

In [ ]:


def predict_next(model, word, sentence, word2idx):


    model.eval()


    h0_predict = torch.zeros(num_layers, 1, hidden_size, , device=device)
    c0_predict = torch.zeros(num_layers, 1, hidden_size, device = device)
    val_per_time_test, _ = (sentence, (h0_predict, c0_predict))
    print(val_per_time_test[-1])
    probs_next_word = val_per_time_test[-1]

    correct_next_word = word
    correct_next_word_id = word2idx[correct_next_word]
    prob_correct = probs_next_word[correct_new_word_id]


    id_for_last = torch.argmax(probs_next_word)
    word_for_last = idx2word[id_for_last]
    print(word_for_last)

    return prob_correct





    #tokens = sentence.split()
    #indices = [word2idx[w] for w in tokens]

    #words_in_sentence = torch.tensor(indices).unsqueeze(0).to(device)

    #batch_size = 1
    #h0 = torch.zeros(num_layers, batch_size, hidden_size).to(device)
    #c0 = torch.zeros(num_layers, batch_size, hidden_size).to(device)

    #prediction, (h, c) = model(words_in_sentence, (h0, c0))
    #print(prediction)
    

predict_next(model, y_train[0], x_train[0], word2idx)


def calculate_perplexity(words):
    # PP = (P (tn | t1,..., t_n-1) ... P(t_3|t_1, t_2)P(t_2|t_1)P(t_1))^{-1/n}
    n = len(words)
    p_inside = 1
    for i in range(len(words), 0, -1):

        p_inside *= predict_next(model, words[i], words[0:i],word2idx)
    PP = (p_inside)**(-1/n)